# Get to Know a Dataset: Africa Brick Kilns

This notebook is a guided tour of the [Africa Brick Kilns](https://registry.opendata.aws/asset-data-africa-brick-kilns/)
dataset, published by [APAD](https://github.com/APAD2024) on the
[Registry of Open Data on AWS](https://registry.opendata.aws/).

APAD (Air Pollution Asset Data) builds openly available, asset-level inventories of
industrial air-pollution sources. This dataset covers **brick kilns** across Africa:
their locations, capacities, fuel types, operating status and — where they have been
derived — estimated annual emissions.

By the end of this notebook you will know how the data is laid out in S3, how to load it,
what the columns mean, what the data can currently support, and where its gaps are.


### Q: How have you organized your dataset? Help us understand the key prefix structure of your S3 bucket.

All APAD asset data lives in a single public bucket, `s3://assetdata-igp/`, in the
`ap-southeast-1` region. Objects are grouped by region of coverage using one top-level prefix
per collection:

```
s3://assetdata-igp/
    africa_assets_csvs/      <- this dataset and the other African sectors
    asset_igp_csvs/          <- the equivalent data for the Indo-Gangetic Plain
```

Within `africa_assets_csvs/`, there is **one CSV per sector**, each a flat table with one row per
asset. This dataset is the object `africa_assets_csvs/brick_kilns_main.csv`.

The collection is small enough (tens to low hundreds of rows per sector) that a single
flat CSV per sector is the most useful layout: it is directly loadable in pandas, R, Excel
or QGIS with no partitioning or query engine required.


In [ ]:
# This notebook requires the following additional libraries
# (install using the preferred method for your environment, e.g. pip or conda):
#
# pandas >= 2.0
# matplotlib >= 3.7
# boto3 >= 1.34

# Built-ins
from io import StringIO

# Installed libraries
import pandas as pd
import matplotlib.pyplot as plt
import boto3
from botocore import UNSIGNED
from botocore.config import Config

BUCKET = "assetdata-igp"
PREFIX = "africa_assets_csvs/"
REGION = "ap-southeast-1"
KEY    = "africa_assets_csvs/brick_kilns_main.csv"

# The bucket is public, so requests are unsigned - no AWS credentials needed.
s3 = boto3.client("s3", region_name=REGION, config=Config(signature_version=UNSIGNED))

Listing the bucket shows the sibling datasets that share this prefix. Each is
documented by its own entry on the Registry of Open Data.

In [ ]:
# List the objects that make up the African asset collection
try:
    listing = s3.list_objects_v2(Bucket=BUCKET, Prefix=PREFIX)
    for item in listing.get("Contents", []):
        print(f"{item['Size']:>10,} bytes   {item['Key']}")
except Exception as exc:
    # Listing requires s3:ListBucket. Individual objects are readable regardless,
    # so the rest of this notebook still runs if listing is unavailable.
    print(f"Could not list bucket ({type(exc).__name__}). Reading the object directly instead.")
    print(f"Dataset object: s3://{BUCKET}/{KEY}")

### Q: What data formats are present in your dataset? What kinds of data are stored using these formats? Can you give any advice for how you work with these data formats?

The dataset is a single **CSV** file, UTF-8 encoded, with a header row and one row per
asset. CSV was chosen deliberately: these are small, wide, human-auditable tables that
researchers and policy analysts frequently open by hand, and CSV imposes no tooling
requirements.

One practical quirk is worth knowing before you parse it:

1. **Some column headers carry a leading space** in this file (`' fuel'`, `' pm10_t_yr'`). Strip whitespace from column names after loading or lookups by name will fail.

The helpers below apply these fixes. They are written to be safe across every dataset in
this collection, so the same code works whichever sector you load.


In [ ]:
def load_apad_csv(bucket: str, key: str) -> pd.DataFrame:
    """Load an APAD asset CSV from public S3 with the dataset's known quirks handled."""
    body = s3.get_object(Bucket=bucket, Key=key)["Body"].read()
    # utf-8-sig transparently strips the byte-order mark if present
    df = pd.read_csv(StringIO(body.decode("utf-8-sig")))
    # Header cells sometimes carry leading/trailing spaces
    df.columns = df.columns.str.strip()
    return df


def to_number(series: pd.Series) -> pd.Series:
    """Coerce a column to numeric, tolerating thousands separators and blanks."""
    return pd.to_numeric(
        series.astype(str).str.replace(",", "", regex=False).str.strip(),
        errors="coerce",
    )


def to_emissions(series: pd.Series) -> pd.Series:
    """Coerce an annual-emissions column to numeric, treating 0 as 'not calculated'.

    A zero in the *_t_yr columns is a placeholder written during inventory
    construction, not a measurement of zero emissions - several assets carrying
    a zero are recorded as actively operating. Summing them as real values would
    understate totals and imply that some countries emit nothing at all, so they
    are converted to NaN and excluded from aggregates.
    """
    return to_number(series).replace(0, pd.NA)

### Q: Can you show us an example of downloading and loading data from your dataset?

Loading the whole dataset is a single call — the file is only a few kilobytes.

If you prefer not to use `boto3`, the object is also readable over plain HTTPS:

```python
df = pd.read_csv("https://assetdata-igp.s3.ap-southeast-1.amazonaws.com/africa_assets_csvs/brick_kilns_main.csv", encoding="utf-8-sig")
df.columns = df.columns.str.strip()
```


In [ ]:
df = load_apad_csv(BUCKET, KEY)

print(f"{df.shape[0]} assets x {df.shape[1]} columns")
df.head()

Every APAD asset dataset shares a common core schema, which makes the sectors
directly comparable:

| Column | Meaning |
|---|---|
| `id` | Stable identifier for the asset within this dataset |
| `name` | Asset or operator name |
| `lat`, `lon` | Location in decimal degrees (WGS 84) |
| `type` | Product recorded for the site (e.g. "brick in ghana") - note this records product and country, not kiln technology |
| `fuel` | Primary fuel or feedstock |
| `region`, `country` | Administrative location |
| `status` | Operating status (e.g. operating, planned, retired) |
| `capacity` | Production or generation capacity |
| `emf*` | Emission factor per unit of activity, by pollutant |
| `*_t_yr` | Derived estimated annual emissions, tonnes/year, by pollutant |
| `source` | Provenance of the record |


**Not every column is populated in every sector.** In this dataset the following are present in the schema but currently empty: `fuel`, `region`, `capacity`, `source`. They are kept so the schema stays consistent across all APAD asset datasets. The completeness check in the next cell shows this directly - always run it before relying on a column.

**Important caveat:** the annual emission columns (`pm25_t_yr`, `pm10_t_yr`, `so2_t_yr`, `nox_t_yr`) are present in the schema but **currently empty** for this dataset. They are retained so the schema stays consistent across all APAD asset datasets, and so they can be populated as emission factors are validated. The analyses below therefore focus on capacity and geographic distribution, which are populated.


In [ ]:
# How complete is each column? This is the first thing to check before analysis.
completeness = (df.notna() & (df.astype(str).apply(lambda c: c.str.strip()) != "")).mean()
completeness.sort_values(ascending=False).apply(lambda v: f"{v:.0%}").to_frame("populated")

### Q: A picture is worth a thousand words. Show us a visual (or several!) from your dataset that either illustrates something informative about your dataset, or that you think might excite someone to dig in further.

Because every record is geolocated, the most immediately useful view is simply *where*
these assets are. The map below plots all 144 assets, coloured by country.
All markers are the same size here, because this dataset does not yet record capacity.

In [ ]:
# Prepare coordinates and capacity
df["lat"] = to_number(df["lat"])
df["lon"] = to_number(df["lon"])
df["capacity_num"] = to_number(df["capacity"])

geo = df.dropna(subset=["lat", "lon"])

fig, ax = plt.subplots(figsize=(10, 9), dpi=100, facecolor="white")

# Scale marker area by capacity; assets with unknown capacity get a small default marker
cap = geo["capacity_num"]
sizes = 30.0 if cap.notna().sum() == 0 else (
    30 + 220 * (cap - cap.min()) / (cap.max() - cap.min())
).fillna(30)

for country, grp in geo.groupby("country"):
    grp_sizes = sizes.loc[grp.index] if hasattr(sizes, "loc") else sizes
    ax.scatter(grp["lon"], grp["lat"], s=grp_sizes, alpha=0.75,
               edgecolor="white", linewidth=0.6, label=country)

ax.set_title("Africa Brick Kilns: asset locations", fontsize=15, fontweight="bold", pad=15)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.grid(True, linestyle="--", alpha=0.3)

# Frame the plot on the data with a small margin, so the assets fill the figure
# rather than being lost in empty ocean.
pad_x = max((geo["lon"].max() - geo["lon"].min()) * 0.08, 1.0)
pad_y = max((geo["lat"].max() - geo["lat"].min()) * 0.08, 1.0)
ax.set_xlim(geo["lon"].min() - pad_x, geo["lon"].max() + pad_x)
ax.set_ylim(geo["lat"].min() - pad_y, geo["lat"].max() + pad_y)
ax.set_aspect("equal", adjustable="box")

ax.legend(title="Country", fontsize=9, loc="best")
plt.tight_layout()
plt.show()

The assets are not evenly spread. Counting by country makes the concentration explicit - though bear in mind this reflects the coverage of the inventory as much as the underlying distribution of assets.

In [ ]:
counts = df["country"].str.strip().value_counts()

fig, ax = plt.subplots(figsize=(10, 5), dpi=100, facecolor="white")
ax.bar(counts.index, counts.values, width=0.6 if len(counts) > 2 else 0.35,
       color="#3498db", edgecolor="white", linewidth=1.2)
ax.set_title("Assets per country", fontsize=15, fontweight="bold", pad=15)
ax.set_ylabel("Number of assets")
ax.grid(True, axis="y", linestyle="--", alpha=0.3)
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

counts

### Q: What is one question that you have answered using these data? Can you show us how you came to that answer?

**What kind of brick and clay production does this inventory actually capture, and how
does that differ between countries?**

This dataset is an inventory of *locations and kinds of production*: `capacity`, `fuel`
and the emission columns are not yet populated (see the completeness table above). What
it does record well is what each site produces and where. Comparing the `type` breakdown
across countries shows what the inventory currently covers - and is the natural starting
point for deciding where to direct capacity and emissions fieldwork next.

In [ ]:
# What does the `type` column actually contain? It records the product and country,
# e.g. "brick in ghana" - not the kiln technology.
df["type_clean"] = df["type"].astype(str).str.strip().str.lower()

# Split the product from the country wording so products can be compared across countries
df["product"] = (df["type_clean"]
                 .str.replace(r"\s+in\s+.*$", "", regex=True)
                 .replace({"": pd.NA}))

breakdown = (
    df.assign(country=df["country"].str.strip())
      .pivot_table(index="product", columns="country", values="id",
                   aggfunc="count", fill_value=0)
)
breakdown["TOTAL"] = breakdown.sum(axis=1)
breakdown.sort_values("TOTAL", ascending=False)

In [ ]:
# Plot the product mix per country
plot_df = breakdown.drop(columns="TOTAL")

fig, ax = plt.subplots(figsize=(10, 5.5), dpi=100, facecolor="white")
bottom = None
for product in plot_df.index:
    vals = plot_df.loc[product]
    ax.bar(plot_df.columns, vals, bottom=bottom, label=product,
           edgecolor="white", linewidth=1.0, width=0.6)
    bottom = vals if bottom is None else bottom + vals

ax.set_title("Recorded production type by country", fontsize=15, fontweight="bold", pad=15)
ax.set_ylabel("Number of sites")
ax.grid(True, axis="y", linestyle="--", alpha=0.3)
ax.legend(title="Product", fontsize=9)
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

Nigeria and Ghana dominate the inventory; the DRC is represented by only a handful of
sites, which is a coverage gap rather than a finding about the DRC. The product mix also
differs by country - Ghana records proportionally more clay production than Nigeria, whose
sites are predominantly brick.

That mix matters for any later emissions work: brick and clay products are fired
differently, so a single emission factor applied across all 144 sites would be misleading.
Any future estimate should be built per product and per country rather than as one
Africa-wide figure.

Note also that `type` records *product and country*, not kiln technology - so this
inventory cannot yet distinguish, say, a fixed chimney kiln from a zigzag kiln, which is
the distinction that most affects emissions.

### Q: What is one unanswered question that you think could be answered using these data? Do you have any recommendations or advice for someone wanting to answer this question?

**Can kiln technology and scale be identified from imagery, so this inventory can support emissions estimates?**

This is the largest dataset in the collection at 144 sites, but it records only
location, product type, country and status. Kiln technology, capacity and fuel - the
inputs any emissions estimate needs - are not present, and collecting them on the
ground across four countries is slow work.

Brick kilns are visually distinctive from above, and their technology and rough scale
are often apparent in high-resolution imagery. A method that infers kiln type and
size from open imagery, validated against a sample, would let this inventory support
emissions estimates without a full field campaign.


**Contributing.** All datasets in this collection share the same schema, so anything you
build here transfers to the other sectors. Corrections, additions and questions are
welcome via [GitHub issues](https://github.com/APAD2024/APAD-Asset-Data/issues).

**Licence and citation.** See this dataset's entry on the
[Registry of Open Data](https://registry.opendata.aws/asset-data-africa-brick-kilns/) for licence terms, and
please cite APAD if you build on this work.
